# Does tainted (leaky) data survive a kernel restart?

This notebook trains a model on **leaky data on purpose**, then shows that after restarting the kernel,
none of it is left in memory.

**How to use it:**
1. Run **Part 1** (the leaky version).
2. **Kernel → Restart** (VS Code: *Restart* button at the top).
3. Run **only Part 2** — do **not** re-run Part 1.

## Part 1 — The leaky version ❌

The classic mistake: fitting the scaler on **all** the data before splitting.
The scaler's averages then include the test rows, so information from the test set leaks into training.
We then **train a model** on that tainted data.

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)

# ❌ LEAK: the scaler sees every row, including the future test rows
tainted_scaler = StandardScaler()
tainted_scaler.fit(X)
X_scaled = tainted_scaler.transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, random_state=42)

# Train a model on the tainted data
tainted_model = LogisticRegression(max_iter=1000)
tainted_model.fit(X_train, y_train)
print(f"Tainted model test accuracy: {tainted_model.score(X_test, y_test):.3f}")

Tainted model test accuracy: 0.979


### How can we *prove* a scaler is tainted?

A `StandardScaler` remembers the average of the data it was fitted on (`scaler.mean_`).
- If it matches the average of **all** the data → it saw the test set → **tainted**.
- If it matches the average of **only the training rows** → **clean**.

In [2]:
def check_scaler(scaler, X_all, X_train_raw):
    if np.allclose(scaler.mean_, X_train_raw.mean(axis=0)):
        return "CLEAN ✅ (fitted on training rows only)"
    if np.allclose(scaler.mean_, X_all.mean(axis=0)):
        return "TAINTED ❌ (fitted on ALL data, including the test set)"
    return "Unknown"

# The raw (unscaled) training rows, using the same split
X_train_raw, _, _, _ = train_test_split(X, y, random_state=42)

print("tainted_scaler is:", check_scaler(tainted_scaler, X, X_train_raw))

tainted_scaler is: TAINTED ❌ (fitted on ALL data, including the test set)


---
## ⚠️ Now restart the kernel

Imagine you've spotted the mistake and fixed your code (Part 2 is the fixed version).

**Kernel → Restart**, then continue below. Don't run the cells above again.

---
## Part 2 — After the restart, with fixed code ✅

### Test 1: Is any of the tainted stuff still in memory?

In [1]:
for name in ["tainted_model", "tainted_scaler", "X_scaled", "X_train", "X_test", "y_train", "y_test", "X", "y"]:
    status = "STILL IN MEMORY ❌" if name in globals() else "gone ✅"
    print(f"{name:15s} {status}")

tainted_model   gone ✅
tainted_scaler  gone ✅
X_scaled        gone ✅
X_train         gone ✅
X_test          gone ✅
y_train         gone ✅
y_test          gone ✅
X               gone ✅
y               gone ✅


In [2]:
try:
    tainted_model.predict
    print("tainted_model still exists -> the kernel was NOT restarted")
except NameError as e:
    print("NameError:", e)

NameError: name 'tainted_model' is not defined


### Test 2: The fixed code — split first, scaler inside a pipeline

Now the scaler is fitted **inside** the pipeline, on training rows only.

In [3]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True)

# ✅ Split FIRST, on the raw data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
print(f"CV accuracy: {scores.mean():.3f} (± {scores.std():.3f})")

pipe.fit(X_train, y_train)
print(f"Clean model test accuracy: {pipe.score(X_test, y_test):.3f}")

CV accuracy: 0.974 (± 0.017)
Clean model test accuracy: 0.979


In [4]:
def check_scaler(scaler, X_all, X_train_raw):
    if np.allclose(scaler.mean_, X_train_raw.mean(axis=0)):
        return "CLEAN ✅ (fitted on training rows only)"
    if np.allclose(scaler.mean_, X_all.mean(axis=0)):
        return "TAINTED ❌ (fitted on ALL data, including the test set)"
    return "Unknown"

clean_scaler = pipe.named_steps["standardscaler"]
print("Scaler inside the pipeline is:", check_scaler(clean_scaler, X, X_train))

Scaler inside the pipeline is: CLEAN ✅ (fitted on training rows only)


## Conclusion

Restarting the kernel wipes **everything** from memory — the tainted model, the tainted scaler and all the leaky data.
Nothing from Part 1 can affect Part 2.

**Fixed code + Restart & Run All = no leakage left in the results.**